# 임베딩

In [4]:
import numpy as np 
from numpy import dot 
from numpy.linalg import norm 
import pandas as pd 
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI 


In [14]:
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

embeddings = OpenAIEmbeddings(
    model="bge-m3",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

print(embeddings.model)

response = client.embeddings.create(
    model="bge-m3",
    input="안녕하세요"
)

query_result = embeddings.embed_query("저는 배가 고파요")
print(len(query_result))
print(query_result)

bge-m3
1024
[0.02190030924975872, 0.009922291152179241, -0.05081392079591751, -0.01938943937420845, -0.007905061356723309, -0.027220429852604866, 0.032535091042518616, 0.013938399963080883, -0.004771048203110695, -0.023543113842606544, 0.00619593495503068, -0.009899257682263851, -0.0020514000207185745, -0.020061137154698372, 0.005997767206281424, -0.013861698098480701, 0.052468232810497284, 0.006975540891289711, 0.00034481214242987335, 0.01814529113471508, 0.003070204984396696, 0.009832351468503475, -0.0064378902316093445, 0.012411495670676231, -0.0021488177590072155, 0.01992213912308216, 0.016875827684998512, -0.00772808538749814, 0.03752541542053223, -0.007816468365490437, 0.0263619776815176, -0.05109492316842079, 0.008686765097081661, -0.06128151714801788, -0.00911761075258255, -0.032902974635362625, 0.003422257024794817, -0.006520051043480635, -0.013702462427318096, 0.06387100368738174, 0.019326074048876762, -0.010031582787632942, 0.0004924284876324236, -0.01681937836110592, 0.0131

In [15]:
data = [
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요',
]
df = pd.DataFrame(data, columns=['text'])
df

,text
0,주식 시장이 급등했어요
1,시장 물가가 올랐어요
2,전통 시장에는 다양한 물품들을 팔아요
3,부동산 시장이 점점 더 복잡해지고 있어요
4,저는 빠른 비트를 좋아해요
5,최근 비트코인 가격이 많이 변동했어요


In [16]:
def get_embedding(text):
    return embeddings.embed_query(text)


df['embedding'] = df.apply(lambda row : get_embedding(row.text), axis=1)


In [17]:
df

,text,embedding
0,주식 시장이 급등했어요,"[-0.02864864282310009, -0.0014646119670942426,..."
1,시장 물가가 올랐어요,"[0.005269182380288839, 0.026660559698939323, -..."
2,전통 시장에는 다양한 물품들을 팔아요,"[0.0003697999636642635, 0.0012548855738714337,..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[-0.007341176737099886, 0.02207113988697529, -..."
4,저는 빠른 비트를 좋아해요,"[-0.01896851323544979, 0.01289202831685543, -0..."
5,최근 비트코인 가격이 많이 변동했어요,"[-0.025856509804725647, -0.00439491868019104, ..."


In [19]:
def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

def return_answer_candidate(df, query):
    query_embedding= get_embedding(query) 
    df["similarity"] = df.embedding.apply(lambda x  : cos_sim(np.array(x), np.array(query_embedding)))
    
    top_three_doc = df.sort_values("similarity", ascending=False).head(3)
    
    return top_three_doc

In [20]:
sim_result = return_answer_candidate(df, "과일 값이 비싸다")
sim_result

,text,embedding,similarity
1,시장 물가가 올랐어요,"[0.005269182380288839, 0.026660559698939323, -...",0.641932
5,최근 비트코인 가격이 많이 변동했어요,"[-0.025856509804725647, -0.00439491868019104, ...",0.603086
0,주식 시장이 급등했어요,"[-0.02864864282310009, -0.0014646119670942426,...",0.576178
